# Faithfulness QA Validation

## 1. Purpose

Faithfulness asks whether materially checkable claims in the generated output are supported by the authoritative context.


## 2. Imports and output location

The helper locates the repository root whether Jupyter starts at the repository root or inside `notebooks/qa`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from idp_eval import EvaluationCase, EvaluationFramework, create_azure_judge
from idp_eval.judges import AzureJudgeConfig

from idp_eval import FaithfulnessEvaluator


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "idp_eval").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the idp-eval repository.")


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "qa_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Configure the judge

Replace every placeholder before running. No Phoenix server is required. If your
application already constructs a compatible judge, you may replace this cell
with that existing construction. Keep credentials in your application's secret
management system rather than saving them in this notebook.


In [ ]:
azure_config = AzureJudgeConfig(
    model="YOUR_AZURE_DEPLOYMENT",
    azure_endpoint="YOUR_AZURE_ENDPOINT",
    tenant_id="YOUR_TENANT_ID",
    client_id="YOUR_CLIENT_ID",
    client_secret="YOUR_CLIENT_SECRET",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)

judge = create_azure_judge(config=azure_config)


## 4. Five mock evaluation cases

These cases intentionally span clear pass, partial, and fail behaviors. `expected_behavior` is a human QA aid, not an exact model-score assertion.


In [ ]:
cases = [
    EvaluationCase(
        case_id="FAITH-001",
        input="Summarize the support plan.",
        context="The plan includes weekday email support and a four-hour response target.",
        output="Customers receive weekday email support with a four-hour response target.",
    ),
    EvaluationCase(
        case_id="FAITH-002",
        input="Summarize the service commitments.",
        context="The service target is 99.9% monthly uptime and email support during business hours.",
        output="The service provides 99.9% monthly uptime and 24/7 phone support.",
    ),
    EvaluationCase(
        case_id="FAITH-003",
        input="Describe the hosting policy.",
        context="Customer data is encrypted at rest and hosted in US regions.",
        output="Customer data has EU-only residency, 99.99% uptime, and 24/7 phone support.",
    ),
    EvaluationCase(
        case_id="FAITH-004",
        input="Summarize account security.",
        context="Administrators must use MFA. Accounts lock after five failed sign-in attempts.",
        output="Administrators use MFA, accounts lock after five failures, and passwords rotate every 30 days.",
    ),
    EvaluationCase(
        case_id="FAITH-005",
        input="Describe the order-processing rules.",
        context={
            "payment": {"methods": ["card", "ACH"], "capture": "at shipment"},
            "shipping": {"regions": ["US", "Canada"], "tracking": True},
        },
        output={
            "payment": "Card and ACH are accepted; payment is captured at shipment.",
            "shipping": "Orders ship within the US and Canada with tracking.",
        },
    ),
]

expected_behavior = {
    "FAITH-001": "fully faithful",
    "FAITH-002": "partially hallucinated — unsupported 24/7 phone support",
    "FAITH-003": "strongly hallucinated — several unsupported claims",
    "FAITH-004": "mixed — two supported claims and one unsupported claim",
    "FAITH-005": "fully grounded structured output",
}


## 5. Inspect the mock inputs


In [ ]:
case_rows = []
for case in cases:
    case_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "input": getattr(case, "input"),
        "context": getattr(case, "context"),
        "output": getattr(case, "output"),
    })

cases_df = pd.DataFrame(case_rows)
display(cases_df)


## 6. Configure one evaluator and Excel output

This notebook runs exactly one metric. `resume=False` creates a fresh QA workbook and no Phoenix tracing is configured.


In [ ]:
excel_path = OUTPUT_DIR / "faithfulness_validation.xlsx"
evaluator = FaithfulnessEvaluator(max_items=None, reason_mode="overall", verbose=True)
framework = EvaluationFramework(
    evaluators=[evaluator],
    judge=judge,
    output="excel",
    excel_path=str(excel_path),
    resume=False,
)

results = framework.evaluate_many(
    cases,
    run_name="qa-validation",
    dataset_name="mock-acceptance-cases",
    show_progress=True,
)


## 7. Result summary


In [ ]:
METRIC_NAME = "faithfulness"
summary_rows = []
for case, result_map in zip(cases, results, strict=True):
    result = result_map[METRIC_NAME]
    summary_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "score": result.score,
        "label": result.label,
        "explanation": result.explanation,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


## 8. Inspect Excel output

The workbook summary is in `evaluations`; item-level evidence is in `faithfulness_items`.


In [ ]:
evaluations_df = pd.read_excel(excel_path, sheet_name="evaluations")
display(evaluations_df)

details_df = pd.read_excel(excel_path, sheet_name="faithfulness_items")
display(details_df)


## 9. Sanity assertions

These assertions validate framework/output behavior and broad direction only; they do not require exact LLM-generated fractions.


In [ ]:
assert len(results) == 5
assert excel_path.exists()
assert all(METRIC_NAME in result_map for result_map in results)
assert len(evaluations_df) == 5
assert set(evaluations_df["key_id"]) == {case.case_id for case in cases}
assert len(details_df) >= 5
assert results[0][METRIC_NAME].score >= results[2][METRIC_NAME].score
print("Faithfulness QA sanity checks passed.")


## 10. Close judge resources and report the workbook path


In [ ]:
judge.close()
print(f"Excel output: {excel_path.resolve()}")
